# Análisis de tráfico del portal Joomla
Este cuaderno se conecta a PostgreSQL (servicio `database`, puerto 5432) y analiza los logs de Nginx expuestos en `monitoring.v_access_logs`.

**Antes de ejecutar:** navega por `http://localhost/` para generar tráfico.

In [5]:
import os
import pandas as pd
import matplotlib.pyplot as plt
from sqlalchemy import create_engine, text
from sqlalchemy.engine import URL

url = URL.create(
    "postgresql+psycopg2",
    username=os.environ["POSTGRES_USER"],
    password=os.environ["POSTGRES_PASSWORD"],
    host=os.environ.get("POSTGRES_HOST", "database"),
    port=5432,
    database=os.environ["POSTGRES_DB"],
)
engine = create_engine(url)

with engine.connect() as conn:
    print(conn.execute(text("SELECT version()")).scalar())

OperationalError: (psycopg2.OperationalError) could not translate host name "database" to address: Name or service not known

(Background on this error at: https://sqlalche.me/e/21/e3q8)

In [ ]:
pd.read_sql("SELECT table_name FROM information_schema.tables WHERE table_schema='public' ORDER BY 1 LIMIT 15", engine)

In [ ]:
df = pd.read_sql("SELECT * FROM monitoring.v_access_logs ORDER BY ts", engine)
print(f"Total de peticiones registradas: {len(df)}")
df.head(10)

In [ ]:
if df.empty:
    print("Sin registros aún: navega por http://localhost/ y vuelve a ejecutar esta celda.")
else:
    fig, ax = plt.subplots(1, 3, figsize=(18, 4))
    df.set_index("ts").resample("1min").size().plot(ax=ax[0], title="Peticiones por minuto")
    df["status"].value_counts().sort_index().plot.bar(ax=ax[1], title="Peticiones por código HTTP")
    df["client_ip"].value_counts().head(10).plot.barh(ax=ax[2], title="Top 10 IPs")
    plt.tight_layout()
    plt.show()